In [3]:
"""
Advanced Missing Data Management Strategy
==========================================

For financial data with missing percentages ranging from 0% to 56%

Strategies:
1. Drop if >40% missing (too unreliable)
2. Keep + Impute if 10-40% missing (critical features)
3. Keep + Flag if 5-10% missing (mostly present)
4. No action if <5% missing (minimal impact)
"""

import pandas as pd
import numpy as np
from typing import Tuple, Dict, List
import warnings
warnings.filterwarnings('ignore')


class MissingDataAnalyzer:
    """
    Analyze and manage missing values in financial dataset
    """
    
    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()
        self.analysis = self._analyze()
        
    def _analyze(self) -> pd.DataFrame:
        """Analyze missing values for numeric columns only"""
    
        # Exclude non-numeric columns
        df_numeric = (
            self.df
            .drop(columns=["ticker", "fiscalDateEnding"], errors="ignore")
            .select_dtypes(include=["number"])
        )
    
        missing_data = pd.DataFrame({
            'column': df_numeric.columns,
            'missing_%': df_numeric.isnull().mean().values * 100,
            'missing_count': df_numeric.isnull().sum().values,
            'non_null_count': df_numeric.notna().sum().values,
            'data_type': df_numeric.dtypes.values,
            'mean': df_numeric.mean().values,
            'std': df_numeric.std().values,
            'min': df_numeric.min().values,
            'max': df_numeric.max().values,
            'skew': df_numeric.skew(),
            'kurtosis': df_numeric.kurt()
        })
    
        return missing_data.sort_values('missing_%', ascending=False)
        
    
    def categorize_by_missingness(self) -> Dict[str, List[str]]:
        """Categorize columns by missingness level"""
        categories = {
            'critical': [],      # >40% missing - DROP or USE WITH CAUTION
            'high': [],          # 20-40% missing - IMPUTE
            'medium': [],        # 10-20% missing - IMPUTE + FLAG
            'low': [],           # 5-10% missing - FLAG ONLY
            'minimal': [],       # <5% missing - KEEP AS IS
        }
        
        for _, row in self.analysis.iterrows():
            pct = row['missing_%']
            col = row['column']
            
            if pct > 40:
                categories['critical'].append(col)
            elif pct > 20:
                categories['high'].append(col)
            elif pct > 10:
                categories['medium'].append(col)
            elif pct > 5:
                categories['low'].append(col)
            else:
                categories['minimal'].append(col)
        
        return categories
    
    def print_summary(self):
        """Print analysis summary"""
        print("\n" + "="*80)
        print("MISSING DATA ANALYSIS SUMMARY")
        print("="*80)
        
        print(f"\nTotal rows: {len(self.df)}")
        print(f"Total columns: {len(self.df.columns)}")
        print(f"Overall missing: {self.df.isnull().sum().sum()} cells ({self.df.isnull().sum().sum()/(len(self.df)*len(self.df.columns))*100:.2f}%)")
        
        categories = self.categorize_by_missingness()
        
        print("\n" + "-"*80)
        print("CATEGORIZATION BY MISSINGNESS LEVEL")
        print("-"*80)
        
        for category, columns in categories.items():
            if columns:
                print(f"\n{category.upper()} MISSING ({len(columns)} columns):")
                for col in columns:
                    pct = self.analysis[self.analysis['column']==col]['missing_%'].values[0]
                    print(f"  • {col:45s} {pct:6.2f}%")
    
    def get_recommendations(self) -> Dict[str, Dict]:
        """Get handling recommendations for each category"""
        categories = self.categorize_by_missingness()
        
        recommendations = {
            'critical': {
                'action': 'DROP or USE WITH EXTREME CAUTION',
                'reason': '>40% missing means data is unreliable',
                'alternative': 'Create alternative features or drop from model',
                'confidence': 'LOW',
                'columns': categories['critical']
            },
            'high': {
                'action': 'IMPUTE + FLAG',
                'reason': '20-40% missing; significant but manageable',
                'method': 'Industry mean, KNN, or domain-specific',
                'confidence': 'MEDIUM',
                'columns': categories['high']
            },
            'medium': {
                'action': 'IMPUTE + FLAG',
                'reason': '10-20% missing; mostly present',
                'method': 'Mean/median imputation recommended',
                'confidence': 'MEDIUM-HIGH',
                'columns': categories['medium']
            },
            'low': {
                'action': 'FLAG + ANALYZE',
                'reason': '5-10% missing; minimal impact',
                'method': 'Simple imputation or forward/backward fill',
                'confidence': 'HIGH',
                'columns': categories['low']
            },
            'minimal': {
                'action': 'KEEP AS IS',
                'reason': '<5% missing; negligible impact',
                'method': 'No action needed; optional flag',
                'confidence': 'VERY HIGH',
                'columns': categories['minimal']
            }
        }
        
        return recommendations



In [4]:
"""
Fixed Missing Data Handler - Handles Mixed Data Types
=======================================================

Problem: Original code tried to impute string columns
Solution: Separate numeric from non-numeric columns
"""

import pandas as pd
import numpy as np
from typing import Tuple, Dict, List
import warnings
warnings.filterwarnings('ignore')


class FixedMissingDataHandler:
    """
    Handle missing values with proper type checking
    
    Separates:
    - Numeric columns (impute with mean/median)
    - String/categorical columns (handle separately)
    - Timestamp columns (keep as-is)
    """
    
    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()
        self.original_df = df.copy()
        self.numeric_cols = None
        self.string_cols = None
        self.datetime_cols = None
        self._classify_columns()
        self.operations_log = []
    
    def _classify_columns(self):
        """Classify columns by data type"""
        self.numeric_cols = []
        self.string_cols = []
        self.datetime_cols = []
        
        for col in self.df.columns:
            dtype = self.df[col].dtype
            
            if pd.api.types.is_numeric_dtype(dtype):
                self.numeric_cols.append(col)
            elif pd.api.types.is_datetime64_any_dtype(dtype):
                self.datetime_cols.append(col)
            else:
                self.string_cols.append(col)
        
        print(f"\nColumn Classification:")
        print(f"  Numeric columns: {len(self.numeric_cols)}")
        print(f"  String columns: {len(self.string_cols)}")
        print(f"  DateTime columns: {len(self.datetime_cols)}")
    
    def handle_string_columns(self):
        """Handle missing values in string columns"""
        print(f"\n{'='*80}")
        print(f"HANDLING STRING/CATEGORICAL COLUMNS")
        print(f"{'='*80}")
        
        for col in self.string_cols:
            missing_count = self.df[col].isnull().sum()
            
            if missing_count == 0:
                print(f"  ✓ {col}: No missing values")
                continue
            
            # For string columns: forward fill or keep as NaN
            # Usually: ticker, currency, etc. shouldn't be imputed
            print(f"  ⚠ {col}: {missing_count} missing values")
            
            # If it's reportedCurrency or similar, fill with mode (most common)
            if 'currency' in col.lower() or 'currency' in col.lower():
                mode_val = self.df[col].mode()
                if len(mode_val) > 0:
                    self.df[col].fillna(mode_val[0], inplace=True)
                    print(f"    → Filled with mode: {mode_val[0]}")
            else:
                # Keep as NaN (don't impute categorical data)
                print(f"    → Keeping as NaN (categorical data)")
        
        self.operations_log.append({
            'operation': 'handle_string_columns',
            'columns': self.string_cols
        })
    
    def handle_datetime_columns(self):
        """Handle missing values in datetime columns"""
        print(f"\n{'='*80}")
        print(f"HANDLING DATETIME COLUMNS")
        print(f"{'='*80}")
        
        for col in self.datetime_cols:
            missing_count = self.df[col].isnull().sum()
            
            if missing_count == 0:
                print(f"  ✓ {col}: No missing values")
            else:
                print(f"  ⚠ {col}: {missing_count} missing (keeping as NaN)")
        
        self.operations_log.append({
            'operation': 'handle_datetime_columns',
            'columns': self.datetime_cols
        })
    
    def drop_critical_columns(self, threshold: float = 0.4) -> pd.DataFrame:
        """
        Drop numeric columns with >threshold missing values
        """
        print(f"\n{'='*80}")
        print(f"DROPPING CRITICAL COLUMNS (>{threshold*100:.0f}% missing)")
        print(f"{'='*80}")
        
        # Only check numeric columns
        missing_pct = self.df[self.numeric_cols].isnull().sum() / len(self.df)
        cols_to_drop = missing_pct[missing_pct > threshold].index.tolist()
        
        if cols_to_drop:
            print(f"Dropping {len(cols_to_drop)} columns:")
            for col in cols_to_drop:
                pct = missing_pct[col] * 100
                print(f"  ✗ {col:45s} {pct:6.2f}%")
            
            self.df.drop(columns=cols_to_drop, inplace=True)
            self.numeric_cols = [c for c in self.numeric_cols if c not in cols_to_drop]
        else:
            print("  No columns to drop")
        
        self.operations_log.append({
            'operation': 'drop_critical',
            'columns': cols_to_drop,
            'threshold': threshold
        })
        
        return self.df
    
    def impute_structural_zero(self, columns: List[str]) -> pd.DataFrame:
        """
        Fill specific numeric columns with zero
        
        Only works on numeric columns
        """
        print(f"\n{'='*80}")
        print(f"IMPUTING STRUCTURAL ZEROS")
        print(f"{'='*80}")
        
        # Filter to only numeric columns that exist
        valid_cols = [c for c in columns if c in self.numeric_cols and c in self.df.columns]
        
        if not valid_cols:
            print("  No valid numeric columns to fill with zero")
            return self.df
        
        print(f"Filling {len(valid_cols)} numeric columns with 0:")
        
        for col in valid_cols:
            missing_count = self.df[col].isnull().sum()
            if missing_count > 0:
                self.df[col].fillna(0.0, inplace=True)
                print(f"  ✓ {col:45s} {missing_count:4d} values → 0")
        
        self.operations_log.append({
            'operation': 'impute_zero',
            'columns': valid_cols
        })
        
        return self.df
    
    def impute_median(self, columns: List[str] = None) -> pd.DataFrame:
        """
        Fill numeric columns with median (robust to outliers)
        
        If columns not specified, uses all remaining numeric columns
        """
        print(f"\n{'='*80}")
        print(f"IMPUTING WITH MEDIAN")
        print(f"{'='*80}")
        
        if columns is None:
            # Use all numeric columns with missing values
            columns = [c for c in self.numeric_cols 
                      if c in self.df.columns and self.df[c].isnull().any()]
        else:
            # Filter to only numeric columns that exist
            columns = [c for c in columns 
                      if c in self.numeric_cols and c in self.df.columns]
        
        if not columns:
            print("  No numeric columns to impute with median")
            return self.df
        
        print(f"Filling {len(columns)} numeric columns with median:")
        
        for col in columns:
            missing_count = self.df[col].isnull().sum()
            if missing_count > 0:
                median_val = self.df[col].median()
                self.df[col].fillna(median_val, inplace=True)
                print(f"  ✓ {col:45s} {missing_count:4d} values → {median_val:12.2f}")
        
        self.operations_log.append({
            'operation': 'impute_median',
            'columns': columns
        })
        
        return self.df
    
    def impute_mean(self, columns: List[str] = None) -> pd.DataFrame:
        """
        Fill numeric columns with mean
        
        If columns not specified, uses all remaining numeric columns
        """
        print(f"\n{'='*80}")
        print(f"IMPUTING WITH MEAN")
        print(f"{'='*80}")
        
        if columns is None:
            # Use all numeric columns with missing values
            columns = [c for c in self.numeric_cols 
                      if c in self.df.columns and self.df[c].isnull().any()]
        else:
            # Filter to only numeric columns that exist
            columns = [c for c in columns 
                      if c in self.numeric_cols and c in self.df.columns]
        
        if not columns:
            print("  No numeric columns to impute with mean")
            return self.df
        
        print(f"Filling {len(columns)} numeric columns with mean:")
        
        for col in columns:
            missing_count = self.df[col].isnull().sum()
            if missing_count > 0:
                mean_val = self.df[col].mean()
                self.df[col].fillna(mean_val, inplace=True)
                print(f"  ✓ {col:45s} {missing_count:4d} values → {mean_val:12.2f}")
        
        self.operations_log.append({
            'operation': 'impute_mean',
            'columns': columns
        })
        
        return self.df
    
    def create_missing_flags(self) -> pd.DataFrame:
        """
        Create flags tracking which values were imputed
        
        Only for numeric columns
        """
        print(f"\n{'='*80}")
        print(f"CREATING MISSING FLAGS")
        print(f"{'='*80}")
        
        flag_count = 0
        
        for col in self.numeric_cols:
            if col in self.df.columns:
                flag_col = f'{col}_was_missing'
                self.df[flag_col] = self.original_df[col].isnull().astype(int)
                
                if self.df[flag_col].sum() > 0:
                    flag_count += 1
        
        print(f"  ✓ Created {flag_count} flag columns for tracking")
        
        self.operations_log.append({
            'operation': 'create_flags',
            'flags_created': flag_count
        })
        
        return self.df
    
    def print_summary(self):
        """Print summary of operations"""
        print(f"\n{'='*80}")
        print(f"MISSING DATA IMPUTATION SUMMARY")
        print(f"{'='*80}")
        
        print(f"\nOriginal Data:")
        print(f"  Rows: {len(self.original_df)}")
        print(f"  Columns: {len(self.original_df.columns)}")
        print(f"  Missing cells: {self.original_df.isnull().sum().sum():,}")
        
        print(f"\nCleaned Data:")
        print(f"  Rows: {len(self.df)}")
        print(f"  Columns: {len(self.df.columns)}")
        print(f"  Missing cells: {self.df.isnull().sum().sum():,}")
        
        if self.df.isnull().sum().sum() == 0:
            print(f"\n✅ DATASET IS 100% COMPLETE!")
        else:
            print(f"\n⚠ Still {self.df.isnull().sum().sum()} missing values")
            print(f"Columns with missing:")
            for col in self.df.columns:
                missing = self.df[col].isnull().sum()
                if missing > 0:
                    print(f"  • {col}: {missing} values")
        
        return True


# ==================== COMPLETE FIXED WORKFLOW ====================

def execute_fixed_strategy(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    """
    Execute complete missing data strategy with proper type handling
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with mixed data types (numeric, string, datetime)
    
    Returns:
    --------
    Tuple[pd.DataFrame, dict] : Cleaned dataframe and log
    """
    
    print("\n" + "█"*80)
    print("█" + " "*15 + "FIXED MISSING DATA STRATEGY (Type-Safe)" + " "*28 + "█")
    print("█"*80)
    
    # Create handler
    handler = FixedMissingDataHandler(df)
    
    # Step 1: Handle non-numeric columns
    print("\n" + "─"*80)
    print("STEP 1: HANDLE NON-NUMERIC COLUMNS")
    print("─"*80)
    handler.handle_string_columns()
    handler.handle_datetime_columns()
    
    # Step 2: Drop critical numeric columns (>40% missing)
    print("\n" + "─"*80)
    print("STEP 2: DROP CRITICAL COLUMNS")
    print("─"*80)
    handler.drop_critical_columns(threshold=0.4)
    
    # Step 3: Impute structural zeros (numeric columns)
    print("\n" + "─"*80)
    print("STEP 3: STRUCTURAL ZEROS")
    print("─"*80)
    zero_columns = [
        'interestIncome', 
        'capitalLeaseObligations', 
        'researchAndDevelopment',
        'shortTermInvestments', 
        'longTermDebt', 
        'shortTermDebt',
        'otherNonCurrentLiabilities',
        'netInterestIncome'
    ]
    # Filter to only existing numeric columns
    zero_columns = [c for c in zero_columns if c in handler.numeric_cols]
    handler.impute_structural_zero(zero_columns)
    
    # Step 4: Impute with median (numeric columns)
    print("\n" + "─"*80)
    print("STEP 4: MEDIAN IMPUTATION FOR ASSETS")
    print("─"*80)
    median_columns = [
        'propertyPlantEquipment',
        'inventory',
        'intangibleAssets',
        'intangibleAssetsExcludingGoodwill',
        'commonStock',
        'goodwill'
    ]
    median_columns = [c for c in median_columns if c in handler.numeric_cols]
    handler.impute_median(median_columns)
    
    # Step 5: Impute remaining with median (safest for numeric)
    print("\n" + "─"*80)
    print("STEP 5: IMPUTE REMAINING NUMERIC WITH MEDIAN")
    print("─"*80)
    # Get all numeric columns with remaining missing
    remaining_numeric = [c for c in handler.numeric_cols 
                        if c in handler.df.columns and handler.df[c].isnull().any()]
    handler.impute_median(remaining_numeric)
    
    # Step 6: Create flags
    print("\n" + "─"*80)
    print("STEP 6: CREATE TRACKING FLAGS")
    print("─"*80)
    handler.create_missing_flags()
    
    # Step 7: Print summary
    print("\n" + "─"*80)
    print("STEP 7: VERIFICATION")
    print("─"*80)
    handler.print_summary()
    
    return handler.df, {
        'operations': handler.operations_log,
        'numeric_cols': handler.numeric_cols,
        'string_cols': handler.string_cols,
        'datetime_cols': handler.datetime_cols
    }


# ==================== USAGE ====================

if __name__ == "__main__":
    
    print("""
USAGE:
======

from fixed_missing_data import execute_fixed_strategy

# Your data (with mixed types: numeric, string, datetime)
df = pd.read_csv('your_data.csv')

# Execute complete strategy
df_clean, log = execute_fixed_strategy(df)

# Save result
df_clean.to_csv('data_clean.csv', index=False)

# Check which columns were imputed
for action in log['operations']:
    if action['operation'] == 'create_flags':
        print(f"Created {action['flags_created']} tracking flags")
    """)


USAGE:

from fixed_missing_data import execute_fixed_strategy

# Your data (with mixed types: numeric, string, datetime)
df = pd.read_csv('your_data.csv')

# Execute complete strategy
df_clean, log = execute_fixed_strategy(df)

# Save result
df_clean.to_csv('data_clean.csv', index=False)

# Check which columns were imputed
for action in log['operations']:
    if action['operation'] == 'create_flags':
        print(f"Created {action['flags_created']} tracking flags")
    


In [5]:

df = pd.read_csv("FinancialDataHandler/combined_data.csv")
df.isna().sum().sum()


21704

In [6]:

missing_pct = (df[df.columns].isna().sum() / len(df)) * 100 
missing_pct[missing_pct > 0].sort_values( ascending=False)


longTermDebtNoncurrent                    100.000000
depreciation                              100.000000
comprehensiveIncomeNetOfTax               100.000000
investmentIncomeNet                       100.000000
interestAndDebtExpense                    100.000000
currentDebt                               100.000000
deferredRevenue                           100.000000
nonInterestIncome                         100.000000
otherNonCurrentAssets                     100.000000
investments                               100.000000
accumulatedDepreciationAmortizationPPE     98.970588
treasuryStock                              80.661765
otherNonOperatingIncome                    56.176471
currentLongTermDebt                        47.132353
longTermInvestments                        41.397059
propertyPlantEquipment                     37.720588
capitalLeaseObligations                    31.691176
interestIncome                             31.176471
inventory                                  21.

In [7]:

# 3. Optional: Drop columns > 70% missing
df = df.loc[:, df.isna().mean() < 0.7]


In [8]:

analyzer = MissingDataAnalyzer(df)
analyzer.print_summary()



MISSING DATA ANALYSIS SUMMARY

Total rows: 1360
Total columns: 53
Overall missing: 5661 cells (7.85%)

--------------------------------------------------------------------------------
CATEGORIZATION BY MISSINGNESS LEVEL
--------------------------------------------------------------------------------

CRITICAL MISSING (3 columns):
  • otherNonOperatingIncome                        56.18%
  • currentLongTermDebt                            47.13%
  • longTermInvestments                            41.40%

HIGH MISSING (5 columns):
  • propertyPlantEquipment                         37.72%
  • capitalLeaseObligations                        31.69%
  • interestIncome                                 31.18%
  • inventory                                      21.76%
  • shortTermInvestments                           20.44%

MEDIUM MISSING (6 columns):
  • longTermDebt                                   17.13%
  • researchAndDevelopment                         14.93%
  • commonStock                

In [9]:

recommendations = get_specific_recommendations()
for category, details in recommendations.items():
    print(category, details)
    

NameError: name 'get_specific_recommendations' is not defined

In [13]:

df_clean, log = execute_fixed_strategy(df)
df_clean.to_csv('data_clean.csv', index=False)



████████████████████████████████████████████████████████████████████████████████
█               FIXED MISSING DATA STRATEGY (Type-Safe)                            █
████████████████████████████████████████████████████████████████████████████████

Column Classification:
  Numeric columns: 49
  String columns: 4
  DateTime columns: 0

────────────────────────────────────────────────────────────────────────────────
STEP 1: HANDLE NON-NUMERIC COLUMNS
────────────────────────────────────────────────────────────────────────────────

HANDLING STRING/CATEGORICAL COLUMNS
  ✓ ticker: No missing values
  ✓ fiscalDateEnding: No missing values
  ⚠ reportedCurrency_x: 28 missing values
    → Filled with mode: USD
  ⚠ reportedCurrency_y: 28 missing values
    → Filled with mode: USD

HANDLING DATETIME COLUMNS

────────────────────────────────────────────────────────────────────────────────
STEP 2: DROP CRITICAL COLUMNS
────────────────────────────────────────────────────────────────────────────────

In [14]:
for action in log['operations']:
    if action['operation'] == 'create_flags':
        print(f"Created {action['flags_created']} tracking flags")


Created 33 tracking flags


In [15]:
df_clean.columns

Index(['Unnamed: 0', 'ticker', 'fiscalDateEnding', 'reportedCurrency_x',
       'totalAssets', 'totalCurrentAssets',
       'cashAndCashEquivalentsAtCarryingValue', 'cashAndShortTermInvestments',
       'inventory', 'currentNetReceivables', 'totalNonCurrentAssets',
       'propertyPlantEquipment', 'intangibleAssets',
       'intangibleAssetsExcludingGoodwill', 'goodwill', 'shortTermInvestments',
       'otherCurrentAssets', 'totalLiabilities', 'totalCurrentLiabilities',
       'currentAccountsPayable', 'shortTermDebt', 'totalNonCurrentLiabilities',
       'capitalLeaseObligations', 'longTermDebt', 'shortLongTermDebtTotal',
       'otherCurrentLiabilities', 'otherNonCurrentLiabilities',
       'totalShareholderEquity', 'retainedEarnings', 'commonStock',
       'commonStockSharesOutstanding', 'reportedCurrency_y', 'grossProfit',
       'totalRevenue', 'costOfRevenue', 'costofGoodsAndServicesSold',
       'operatingIncome', 'sellingGeneralAndAdministrative',
       'researchAndDevelopment

In [12]:
'''
Do NOT Start With

Deep learning (overkill for fundamentals)

LSTM (useless for slow-moving accounting data)

Random Forest (weaker than boosting)

SVM (hard to scale)
'''

'\nDo NOT Start With\n\nDeep learning (overkill for fundamentals)\n\nLSTM (useless for slow-moving accounting data)\n\nRandom Forest (weaker than boosting)\n\nSVM (hard to scale)\n'